## Autograd 边界练习

`permute` 可能产生非连续 Tensor；`view` 要求兼容的内存布局，必要时先调用 `contiguous()`。`clone` 复制数据，`detach` 切断计算图。


In [1]:
import torch
base = torch.arange(12., requires_grad=True).reshape(3, 4)
transposed = base.transpose(0, 1)
try:
    transposed.view(-1)
except RuntimeError as error:
    print('expected non-contiguous error:', type(error).__name__)
view = transposed.contiguous().view(-1)
detached = view.detach()
copied = view.clone()
assert not detached.requires_grad and copied.requires_grad
vector = torch.arange(3., requires_grad=True)
vector.square().backward(torch.ones(3))
print('non-scalar gradient:', vector.grad)
assert torch.equal(vector.grad, 2 * vector.detach())

expected non-contiguous error: RuntimeError
non-scalar gradient: tensor([0., 2., 4.])


# 张量、形状与设备

## 学习目标

能够解释 shape、stride、dtype、广播和设备移动，并在运行前预测常见张量操作的输出。完成本章后，你应能读懂后续 CNN、RNN 和 attention 中的维度变化。

## 概念模型与执行路径

张量不是多维列表，而是由数据、形状、数据类型、布局和设备共同定义的对象。先问五个问题：数据有几维？每一维代表什么？元素是什么类型？内存如何排列？计算发生在哪里？

本章按以下顺序建立模型：创建与形状变换 -> 索引和掩码 ->逐元素运算与归约 -> 广播 -> 矩阵乘法 -> 维度重排与拼接 -> 视图和设备。每个实验都先预测，再运行验证。

## 实验 1：创建张量并读取属性

`shape` 描述每一维大小，`stride` 描述沿某一维移动一个元素需要跨过多少存储位置。训练代码中，`(batch, features)`、`(batch, channels, height, width)` 和 `(batch, time, features)` 是最常见的维度语义。

In [2]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / '07-deep-learning/pytorch']
PYTORCH_ROOT = next(path for path in candidates if (path / 'common').exists())
if str(PYTORCH_ROOT) not in sys.path:
    sys.path.insert(0, str(PYTORCH_ROOT))

import torch
from common.runtime import choose_device, seed_everything


In [3]:
seed_everything(42)
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)
print('shape:', x.shape, 'ndim:', x.ndim, 'stride:', x.stride())
print('dtype:', x.dtype, 'device:', x.device)
print(x)


shape: torch.Size([3, 4]) ndim: 2 stride: (4, 1)
dtype: torch.float32 device: cpu
tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])


## 实验 2：形状变换，不改变元素总数

`reshape` 重新解释形状，`flatten` 合并维度，`unsqueeze` 插入长度为 1 的维度。形状变换必须保持元素总数不变；`squeeze` 则可能意外删除 batch 维，因此生产代码常指定 `dim`。

In [6]:
image_batch = torch.arange(24).reshape(2, 3, 4)
print('image_batch:', image_batch.shape)
print('flatten each sample:', image_batch.flatten(start_dim=1).shape)
print('add channel:', image_batch.unsqueeze(1).shape)
print('remove only channel:', image_batch.unsqueeze(1).squeeze(1).shape)


image_batch: torch.Size([2, 3, 4])
flatten each sample: torch.Size([2, 12])
add channel: torch.Size([2, 1, 3, 4])
remove only channel: torch.Size([2, 3, 4])


## 实验 3：索引、切片与布尔掩码

索引是在选择数据，不是在改变数据的含义。切片保留未指定的维度；布尔掩码会把满足条件的元素收集成一维结果，不能假设它保留原形状。

In [7]:
scores = torch.tensor([[0.2, 0.8, 0.1], [0.6, 0.3, 0.9]])
print('one sample:', scores[0].shape)
print('one class across batch:', scores[:, 1].shape)
print('first two classes:', scores[:, :2].shape)
mask = scores > 0.5
print('mask:', mask)
print('selected values:', scores[mask], 'shape:', scores[mask].shape)


one sample: torch.Size([3])
one class across batch: torch.Size([2])
first two classes: torch.Size([2, 2])
mask: tensor([[False,  True, False],
        [ True, False,  True]])
selected values: tensor([0.8000, 0.6000, 0.9000]) shape: torch.Size([3])


## 实验 4：逐元素运算与归约

逐元素运算通常保持形状；归约操作沿指定 `dim` 消除一个维度。`keepdim=True` 会保留长度为 1 的维度，便于后续广播。

In [ ]:
batch_features = torch.tensor([[1., 2., 3.], [4., 5., 6.]])
print('double:', (batch_features * 2).shape)
print('mean over features:', batch_features.mean(dim=1).shape)
print('mean with keepdim:', batch_features.mean(dim=1, keepdim=True).shape)
centered = batch_features - batch_features.mean(dim=1, keepdim=True)
print('centered row means:', centered.mean(dim=1))


## 实验 5：广播

广播从最后一维向前对齐：两个维度相等，或其中一个为 1，或其中一个不存在时才能匹配。广播不复制数据的概念副本，但结果通常会创建新张量。

In [ ]:
x = torch.arange(12, dtype=torch.float32).reshape(3, 4)
bias = torch.tensor([10.0, 20.0, 30.0, 40.0])
print('x + (4,):', (x + bias).shape)
column = torch.tensor([[100.0], [200.0], [300.0]])
print('x + (3, 1):', (x + column).shape)
try:
    x + torch.ones(2)
except RuntimeError as error:
    print('expected broadcasting error:', str(error).splitlines()[0])


## 实验 6：矩阵乘法与批量矩阵乘法

`@` 的关键规则是：左侧最后一维必须等于右侧倒数第二维，输出保留批次维并把最后两维变成 `(左行数, 右列数)`。这正是线性层和 attention 计算的形状基础。

In [ ]:
features = torch.randn(2, 3, 4)  # batch, time, feature
weight = torch.randn(4, 5)
projected = features @ weight
print('(2, 3, 4) @ (4, 5):', projected.shape)
q = torch.randn(2, 3, 4)
k = torch.randn(2, 5, 4)
attention_scores = q @ k.transpose(-2, -1)
print('Q @ K^T:', attention_scores.shape)


## 实验 7：维度重排、拼接与堆叠

`transpose`/`permute` 改变维度顺序；`cat` 在已有维度上连接，`stack` 新增一个维度。图像常用 NCHW，序列常用 batch-first 的 NTF，转换时必须明确每个字母的含义。

In [ ]:
nhwc = torch.randn(2, 28, 28, 3)
nchw = nhwc.permute(0, 3, 1, 2)
print('NHWC -> NCHW:', nchw.shape)
left, right = torch.zeros(2, 3), torch.ones(2, 2)
print('cat features:', torch.cat([left, right], dim=1).shape)
print('stack samples:', torch.stack([left, left]).shape)


## 实验 8：视图、复制与连续布局

切片和 `permute` 可能返回共享存储的视图。`clone` 才会复制数据；`view` 要求 stride 兼容，`reshape` 必要时会复制。不要把“看起来是新变量”误认为“拥有独立数据”。

In [ ]:
base = torch.arange(12, dtype=torch.float32).reshape(3, 4)
view = base[:, :2]
clone = view.clone()
view[0, 0] = -99
print('view changes base:', base[0, 0].item())
print('clone keeps old value:', clone[0, 0].item())
transposed = base.t()
print('transposed contiguous:', transposed.is_contiguous())
print('contiguous copy:', transposed.contiguous().is_contiguous())


## 实验 9：设备移动与 NumPy 边界

`.to(device)` 返回位于目标设备的新张量，不会原地修改原变量。模型参数和输入必须在同一设备；NumPy 只能直接接收 CPU 张量。

In [ ]:
device = choose_device('auto')
x_device = base.to(device)
print('selected device:', device)
print('original:', base.device, 'moved:', x_device.device)
print('back on CPU as NumPy:', x_device.cpu().numpy().shape)


## 底层机制

stride 是张量布局的线索：连续的二维张量 `(3, 4)` 通常有 stride `(4, 1)`。转置只改变解释方式，可能不搬动数据，因此后续 `view` 需要先 `contiguous()`。广播按末尾维度对齐；归约改变维度语义；设备移动会复制数据并产生同步成本。训练热路径中应减少不必要的 `permute`、`.cpu()` 和设备往返。

## 检查点

在不运行代码的情况下回答：

1. `(2, 3, 4) + (4,)` 的输出形状是什么？
2. `(2, 3, 4) @ (4, 5)` 的输出形状是什么？
3. `torch.randn(8, 28, 28, 3).permute(0, 3, 1, 2)` 的输出形状是什么？
4. `scores.mean(dim=1, keepdim=True)` 为什么比 `scores.mean(dim=1)` 更容易与原张量相减？

## 试一试

1. 把 `bias` 改成 shape 为 `(3, 1)` 的张量，解释为什么仍能广播；再尝试 `(2,)`，记录维度错误。
2. 创建 `(4, 3, 28, 28)` 的图像 batch，分别取第 2 张图、第 1 个通道和中心 10x10 区域，写出每一步的 shape。
3. 用 `keepdim=True` 计算每个样本的均值和标准差，并把 batch 标准化到近似均值 0。
4. 预测 `q @ k.transpose(-2, -1)` 的形状，然后改变 query token 数和 key token 数验证预测。
5. 解释为什么 `permute` 后直接 `view` 可能失败，并用 `contiguous()` 修复。

## 常见错误与调试

混用 float32/float64；把输入留在 CPU 而模型在 CUDA；把 batch、channel、time 混为一谈；误以为切片一定复制；使用无 `dim` 的 `squeeze()` 意外删除 batch 维；忽略 `keepdim` 导致广播方向错误；把 `transpose` 后的非连续张量直接 `view`；在 attention 中忘记转置 key 的最后两维。调试时先同时打印 `shape`、`dtype`、`device` 和 `is_contiguous()`。